In [7]:
import importlib
import sys
import os
import torch
import numpy as np
from tqdm.notebook import tqdm
import torch

sys.path.insert(0, '..')
sys.path.insert(1, '../../..')
sys.path.insert(0, "../../src")  # src package


## Generate Train, Val, Test

In [22]:
from perturbation_logic.activity_pertubator import (
    split_prefix_suffix_readable,
    redo_last_activity_of_prefix)
from event_log_loader_service.event_log_loader import (get_train_test_val_datasets,
                                                       extract_feature_info)
np.random.seed(17)

csv_path="../../../data/helpdesk.csv"

properties = {
    'case_name' : 'Case ID',
    'concept_name' : 'Activity',
    'timestamp_name' : 'Complete Timestamp',
    'date_format' : '%Y/%m/%d %H:%M:%S.%f',
    'time_since_case_start_column' : 'case_elapsed_time',
    'time_since_last_event_column' : 'event_elapsed_time',
    'day_in_week_column' : 'day_in_week',
    'seconds_in_day_column' : 'seconds_in_day',
    'min_suffix_size' : 5,
    'train_validation_size' : 0.15,
    'test_validation_size' : 0.2,
    'window_size' : 'auto',
    'categorical_columns' : ['Activity', 'Resource', 'Variant index', 'seriousness', 'customer', 'product', 'responsible_section', 'seriousness_2', 'service_level', 'service_type', 'support_section', 'workgroup'],
    'continuous_columns' : ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day', ],
    'continuous_positive_columns' : [],
}


train_df, val_df, test_df,  = get_train_test_val_datasets(csv_path, properties)


print(len(train_df))

# data_train = split_prefix_suffix_readable(
#     train_df,
#     case_column=properties["case_name"],
#     activity_column=properties["concept_name"],
#     min_suffix_size=0,
# )

data_val = split_prefix_suffix_readable(
    val_df,
    case_column=properties["case_name"],
    activity_column=properties["concept_name"],
    min_suffix_size=2,
)

# data_test = split_prefix_suffix_readable(
#     test_df,
#     case_column=properties["case_name"],
#     activity_column=properties["concept_name"],
#     min_suffix_size=0,
# )

torch.save(data_train, '../../../perturbed_data/helpdesk/train.pkl')
torch.save(data_val, '../../../perturbed_data/helpdesk/val.pkl')
torch.save(data_test, '../../../perturbed_data/helpdesk/test.pkl')


#display(train_df)

#extract feature info
feature_info = extract_feature_info(val_df, properties)


28685


# Create perturbed Datasets

In [27]:
# Last Event Attack
from perturbation_logic.feature_attacks import last_event_attack

# Reset data_val_copy for feature attacks
data_val_copy = data_val.copy()

# Attacks the last event of each prefix
data_pert_last = last_event_attack(
    data=data_val_copy,
    properties=properties,
    feature_info=feature_info,
    attackable_features=['Resource'],
    num_of_features_to_attack=1,
    magnitude=0.5,
    feature_range_scope='local',
    random_seed=17
)

torch.save(data_pert_last, '../../../perturbed_data/helpdesk/last_event_attack_all.pkl')


In [28]:
# Random Event Attack
from perturbation_logic.feature_attacks import random_event_attack

# Attacks random events in each prefix with probability p
data_val_copy = data_val.copy()  
data_pert_random = random_event_attack(
    data=data_val_copy,
    properties=properties,
    feature_info=feature_info,
    attackable_features=['Resource'],
    num_of_features_to_attack=5,
    event_attack_probability=1.0,
    magnitude=0.5,
    feature_range_scope='local',
    random_seed=17
)

torch.save(data_pert_random, '../../../perturbed_data/helpdesk/random_event_attack_all.pkl')


In [5]:
# Apply "redo last activity" augmentation to each prefix/suffix pair

data_val_copy = data_val.copy()  # Reset again
data_pert = {}
for key, (prefix_df, suffix_df) in data_val_copy.items():
    new_prefix, new_suffix = redo_last_activity_of_prefix(
        prefix_df,
        suffix_df,
        properties=properties,
    )
    data_pert[key] = (new_prefix, new_suffix)


torch.save(data_pert, '../../../perturbed_data/helpdesk/redo_pert.pkl')


In [ ]:
# Apply " loop augmentation"
from perturbation_logic.structural_attacks import generate_loop_augmentation

val_loops_clean, val_loops_pert = generate_loop_augmentation(
    val_df,
    properties,
    min_suffix_size=5,
    max_matches_per_loop=3,
    save_path="../../../perturbed_data/helpdesk",
)

# Save final results
torch.save(val_loops_clean, "../../../perturbed_data/helpdesk/loop_augmentation_clean_new.pkl")
torch.save(val_loops_pert, "../../../perturbed_data/helpdesk/loop_augmentation_pert_new.pkl")
print(f"Loop augmentation: {len(val_loops_clean)} clean/pert pairs") 

Loop augmentation: 50 clean/pert pairs


In [ ]:
# # Summarize loop augmentation results into one combined dataset
# import pickle
# import os

# def summarize_loop_augmentation_results(save_path, load_from_disk=False):
#     """
#     Merge val_loops_clean and val_loops_pert into one combined view.
#     If load_from_disk=True, loads from save_path/val_loops_clean.pkl and val_loops_pert.pkl.
#     Otherwise expects val_loops_clean and val_loops_pert to exist in the notebook namespace.
#     Returns (val_loops_clean_merged, val_loops_pert_merged) as dicts.
#     """
#     if load_from_disk and save_path:
#         clean_path = os.path.join(save_path, "val_loops_clean.pkl")
#         pert_path = os.path.join(save_path, "val_loops_pert.pkl")
#         with open(clean_path, "rb") as f:
#             val_loops_clean_merged = pickle.load(f)
#         with open(pert_path, "rb") as f:
#             val_loops_pert_merged = pickle.load(f)
#     else:
#         val_loops_clean_merged = val_loops_clean.copy()
#         val_loops_pert_merged = val_loops_pert.copy()
#     return val_loops_clean_merged, val_loops_pert_merged

# # Use in-memory results from previous cell (or set load_from_disk=True to load from save_path)
# val_loops_clean_combined, val_loops_pert_combined = summarize_loop_augmentation_results(
#     "../../../perturbed_data/helpdesk", load_from_disk=False
# )
# print(f"Combined clean: {len(val_loops_clean_combined)} entries")
# print(f"Combined pert:  {len(val_loops_pert_combined)} entries")

Combined clean: 130 entries
Combined pert:  130 entries


# Compare changes

In [25]:
# Compare clean and perturbed datasets
from perturbation_logic.attack_impact_analyzer import highlight_feature_attack_impact

# Compare clean dataset with random event attack
highlight_feature_attack_impact(
    clean_data_path='../../../perturbed_data/helpdesk/val.pkl',
    perturbed_data_path='../../../perturbed_data/helpdesk/last_event_attack_all.pkl',
    properties=properties
)

Loading clean dataset from: ../../../perturbed_data/helpdesk/val.pkl
Loading perturbed dataset from: ../../../perturbed_data/helpdesk/last_event_attack_all.pkl

Clean dataset has 1898 cases
Perturbed dataset has 1898 cases

COMPARISON RESULTS

Case: Case 1, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    Resource:
      Clean:    Value 1
      Perturbed: Value 19 [CHANGED]


Case: Case 1, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    Resource:
      Clean:    Value 1
      Perturbed: Value 18 [CHANGED]


Case: Case 1, Prefix Length: 3
  [CHANGED] Found 1 event(s) with differences

  Event 2:
    Resource:
      Clean:    Value 2
      Perturbed: Value 7 [CHANGED]


Case: Case 10, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    Resource:
      Clean:    Value 2
      Perturbed: Value 12 [CHANGED]


Case: Case 10, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    Re

In [26]:
# Compare clean and perturbed datasets
from perturbation_logic.attack_impact_analyzer import highlight_structural_attack_impact

# Compare clean dataset with random event attack
highlight_structural_attack_impact(
    # clean_data_path='../../../perturbed_data/helpdesk/loop_augmentation_clean.pkl',
    # perturbed_data_path='../../../perturbed_data/helpdesk/loop_augmentation_pert.pkl',
    clean_data_path='../../../perturbed_data/helpdesk/val.pkl',
    perturbed_data_path='../../../perturbed_data/helpdesk/last_event_attack_all.pkl',
    properties=properties
)

Loading clean dataset from: ../../../perturbed_data/helpdesk/val.pkl
Loading perturbed dataset from: ../../../perturbed_data/helpdesk/last_event_attack_all.pkl

Clean dataset has 1898 cases
Perturbed dataset has 1898 cases

STRUCTURAL ATTACK COMPARISON RESULTS

Case: Case 1, Prefix Length: 1
Prefix
Activity_seq_clean = ['Assign seriousness']
Activity_seq_pert = ['Assign seriousness']
Case_elapsed_time_clean = [0.0]
Case_elapsed_time_pert = [0.0]
Event_elapsed_time_clean = [nan]
Event_elapsed_time_pert = [nan]
Day_in_week_clean = [1.0]
Day_in_week_pert = [1.0]
Seconds_in_day_clean = [53417.0]
Seconds_in_day_pert = [53417.0]

===
suffix
Activity_seq_clean = ['Take in charge ticket', 'Take in charge ticket', 'Resolve ticket', 'Closed', 'EOS', 'EOS', 'EOS', 'EOS', 'EOS']
Activity_seq_pert = ['Take in charge ticket', 'Take in charge ticket', 'Resolve ticket', 'Closed', 'EOS', 'EOS', 'EOS', 'EOS', 'EOS']
Case_elapsed_time_clean = [44.0, 259959.0, 1371849.0, 2671462.0, nan, nan, nan, nan, nan

In [30]:
import pandas as pd

# Merge train_df, val_df, test_df
merged_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print("Combining train_df, val_df, and test_df...")
print(f"Total events: {len(merged_df)}")
print(f"Total cases: {merged_df[properties['case_name']].nunique()}")

# Define feature columns (categorical + continuous)
case_col = properties['case_name']
feature_cols = [
    c for c in
    properties['categorical_columns']
    + properties['continuous_columns']
    + properties.get('continuous_positive_columns', [])
    if c in merged_df.columns
]
total_cases = merged_df[case_col].nunique()
categorical_cols = set(properties['categorical_columns'])

print("\nGrouping events by case...")
print(f"Found {total_cases} unique cases")
print("\nAnalyzing attributes for dynamic behavior...")

# Dynamic feature: varies within at least one case (nunique > 1 per case)
dynamic_results = []
static_features = []

for col in feature_cols:
    def varies_in_case(x, is_cat):
        valid = x.dropna()
        # EOS does not count as change - exclude from distinct-value check
        if is_cat:
            valid = valid[valid.astype(str) != 'EOS']
        return valid.nunique() > 1 if len(valid) > 0 else False

    cases_with_change = merged_df.groupby(case_col)[col].apply(
        lambda x: varies_in_case(x, col in categorical_cols)
    ).sum()
    pct = 100 * cases_with_change / total_cases
    is_cat = col in categorical_cols

    if cases_with_change > 0:
        dynamic_results.append((col, int(cases_with_change), pct, is_cat))
    else:
        static_features.append((col, is_cat))

# Print report
print("\n" + "=" * 80)
print("DYNAMIC ATTRIBUTES ANALYSIS")
print("=" * 80)
print(f"\nTotal cases analyzed: {total_cases}")
print(f"Total attributes checked: {len(feature_cols)}")
print(f"\nFound {len(dynamic_results)} dynamic attributes:\n")

# Example case: first case that has multiple events
example_case = merged_df[case_col].iloc[0]

for col, num_varies, pct, is_cat in dynamic_results:
    feat_type = "Categorical" if is_cat else "Continuous"
    print(f"{col} ({feat_type})")
    print(f"  Varies in {num_varies}/{total_cases} cases ({pct:.1f}%)")

    # Example values from first case
    case_vals = merged_df.loc[merged_df[case_col] == example_case, col].dropna()
    vals_list = case_vals.tolist()
    vals_str = str(vals_list)
    if len(vals_str) > 80:
        vals_str = vals_str[:77] + "..."
    print(f"  Example from case {example_case}:")
    print(f"    Values: {vals_str}")
    print()

print("=" * 80)
print("\nStatic attributes (do not change within cases):")
for col, is_cat in static_features:
    feat_type = "Categorical" if is_cat else "Continuous"
    print(f"  - {col} ({feat_type})")
print("=" * 80)

Combining train_df, val_df, and test_df...
Total events: 44248
Total cases: 4580

Grouping events by case...
Found 4580 unique cases

Analyzing attributes for dynamic behavior...

DYNAMIC ATTRIBUTES ANALYSIS

Total cases analyzed: 4580
Total attributes checked: 16

Found 15 dynamic attributes:

Activity (Categorical)
  Varies in 4580/4580 cases (100.0%)
  Example from case Case 2441:
    Values: ['Assign seriousness', 'Assign seriousness', 'Take in charge ticket', 'Resolv...

Resource (Categorical)
  Varies in 4580/4580 cases (100.0%)
  Example from case Case 2441:
    Values: ['Value 2', 'Value 2', 'Value 2', 'Value 2', 'Value 5', 'EOS', 'EOS', 'EOS', ...

seriousness (Categorical)
  Varies in 4580/4580 cases (100.0%)
  Example from case Case 2441:
    Values: ['Value 1', 'Value 1', 'Value 1', 'Value 1', 'Value 1', 'EOS', 'EOS', 'EOS', ...

customer (Categorical)
  Varies in 4580/4580 cases (100.0%)
  Example from case Case 2441:
    Values: ['Value 186', 'Value 186', 'Value 186', 'Va